In [29]:
import embedding
from transformers import BertTokenizer

In [30]:
import os
os.environ['CURL_CA_BUNDLE'] = ''

In [31]:
 !pip install requests==2.27.1   


[notice] A new release of pip available: 22.3.1 -> 23.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
import pickle
import torch
with open("C:\\Users\\Neela\\Documents\\GitHub\\EntityAspectLinking\\picklefiles\\final_eal.pkl", 'rb') as eal:
    data = pickle.load(eal)

ent = [data[i][0] for i in range(len(data))]

asp = [data[i][1] for i in range(len(data))]

In [33]:
ent[0]

{'id': '0',
 'target_entity': 'Gautama Buddha',
 'paragraph': "Srivastava's discovery of the terracotta sealings bearing the name Kapilavastu has led some scholars to believe that modern-day Piprahwa was the site of the ancient city of Kapilavastu, the capital of the Shakya kingdom, where Siddhartha Gautama spent the first 29 years of his life. Others suggest that the original site of Kapilavastu is located 16 km to the northwest, at Tilaurakot, in what is currently Kapilvastu District in Nepal. This question is especially important to scholars of Buddhist history, as Kapilavastu was the capital of the Shakya kingdom. King Śuddhodana and Queen Māyādevī lived at Kapilavastu, as did their son Prince Siddhartha Gautama until he left the palace at 29 years of age.",
 'entities': [{'eid': '00',
   'entity': 'Kapilavastu (ancient city)',
   'mention': 'Kapilavastu'},
  {'eid': '01', 'entity': 'Shakya', 'mention': 'Shakya'},
  {'eid': '02', 'entity': 'Gautama Buddha', 'mention': 'Siddhartha G

In [34]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
target_emb = torch.zeros(len(ent), 768)
pretrained = 'bert-base-uncased'
ent_emb = embedding.EntityEmbedding(pretrained = pretrained)
for i in range(len(ent)):
    entity_word = ent[i]['target_entity']
    tokens = tokenizer.tokenize(entity_word)
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    input_ids = torch.tensor(input_ids).unsqueeze(0) 
    target_emb[i] = ent_emb(input_ids)
    
    

SSLError: HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /bert-base-uncased/resolve/main/vocab.txt (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:997)')))

In [ ]:
count = 0
for item in ent:
    count += len(item['entities'])
    
count

In [ ]:
t_ent_emb = torch.zeros(count, 768)

for i in range(len(ent)):
    for el in ent[i]['entities']:
        word = el['entity']
        tokens = tokenizer.tokenize(word)
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        input_ids = torch.tensor(input_ids).unsqueeze(0)
        t_ent_emb[i] = ent_emb(input_ids)

In [ ]:
asp_emb = torch.zeros(len(asp), 768)
for i in range(len(asp)):
    aspect = asp[i]['true_aspect']
    tokens = tokenizer.tokenize(aspect)
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    input_ids = torch.tensor(input_ids).unsqueeze(0) 
    asp_emb[i] = ent_emb(input_ids)

In [ ]:
asp[0]

In [ ]:
asp_count = 0
for item in asp:
    for el in item['candidate_aspects']:
        asp_count += len(el['entities'])
asp_count

In [ ]:
a_ent_emb = torch.zeros(asp_count, 768)
for i in range(len(asp)):
    for el in asp[i]['candidate_aspects']:
        for ent in el['entities']:
            word = ent['entity_name']
            tokens = tokenizer.tokenize(word)
            input_ids = tokenizer.convert_tokens_to_ids(tokens)
            input_ids = torch.tensor(input_ids).unsqueeze(0) 
            a_ent_emb[i] = ent_emb(input_ids)
            

In [ ]:
dic = {}
i=0
for item in data:
    _, target = item
    for l in target["candidate_aspects"]:
        for el in l["entities"]:
            dic[el['eid']] = i
            i+= 1